In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# Majority per question baseline

Predict the most frequent training answer for each question_id.


In [2]:
import sys
import json
from pathlib import Path

import pandas as pd


In [3]:
from pathlib import Path
import os

def find_imageclef_root() -> Path:
    env_root = os.environ.get("IMAGECLEF_MEDVQA_GI_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"IMAGECLEF_MEDVQA_GI_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "ImageCLEF_MEDVQA_GI_2023" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "ImageCLEF_MEDVQA_GI_2023" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate ImageCLEF_MEDVQA_GI_2023 root. "
        "Run from within the ImageCLEF_MEDVQA_GI_2023 folder or set IMAGECLEF_MEDVQA_GI_ROOT."
    )

ROOT = find_imageclef_root()
sys.path.append(str(ROOT))

from common import (
    find_long_table,
    load_long_table,
    load_label_maps,
    add_label_ids,
    compute_metrics_per_question,
    compute_binary_metrics,
    save_metrics,
    save_predictions,
    normalize_answer,
)

DATA_PATH = find_long_table(ROOT)
LABEL_MAP_DIR = ROOT / "0_dataset_prep" / "out" / "label_maps"
OUT_DIR = ROOT / "1_baselines" / "out" / "01_majority_per_question"
MODEL_NAME = "majority_per_question"


In [4]:
# Load data
label_maps = load_label_maps(LABEL_MAP_DIR)
long_df = load_long_table(DATA_PATH)

# Majority per question_id from train split
train_df = long_df[long_df["split"] == "train"].copy()
majority = (
    train_df.groupby("question_id")["answer_norm"]
    .apply(lambda s: s.value_counts().idxmax())
    .to_dict()
)


In [5]:
# Apply to all splits
pred_df = long_df.copy()
pred_df["pred_answer"] = pred_df["question_id"].map(majority).map(normalize_answer)

# Map to label ids
pred_df = add_label_ids(pred_df, label_maps, ans_col="answer_norm", out_col="label_id")
pred_df = add_label_ids(pred_df, label_maps, ans_col="pred_answer", out_col="pred_label_id")

# Evaluate per split
for split in sorted(pred_df["split"].unique()):
    df_split = pred_df[pred_df["split"] == split]
    overall, per_q = compute_metrics_per_question(df_split, "label_id", "pred_label_id")
    binary = compute_binary_metrics(df_split, label_maps, "label_id", "pred_label_id")
    split_out = OUT_DIR / split
    save_metrics(split_out, overall, per_q, binary)
    save_predictions(
        df_split,
        split_out,
        columns=["image_id", "question_id", "answer_norm", "pred_answer", "split"],
    )

OUT_DIR


PosixPath('/home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/ImageCLEF_MEDVQA_GI_2023/1_baselines/out/01_majority_per_question')